# Query generator trial (decision 0021)

Two open-weight models each write a backstory and a query for the same 30 grid cells
(`data/queries/generator_trial_cells_v1.jsonl`: 10 English, 10 Hindi, 10 Kannada): Qwen3-14B and Sarvam-M
(24B, built on Mistral Small and post-trained on Indian languages). Both are Apache-2.0 and not gated. Gemma is
left out because Gemma 4 is the relevance judge (decision 0022). The author then rates the outputs blind to the
model, and the model with the better Hindi and Kannada ratings generates the benchmark.

**Before running**
- Runtime: GPU. Colab's T4 or Kaggle's two T4s both work; models load in 4-bit, one at a time, and each model's
  download is deleted before the next one, so the disk holds one model at a time (about 30 GB and 47 GB).
- No token is needed. A Hugging Face token saved as the secret `HF_TOKEN` (Colab) only speeds up downloads.

**Output**: `generator_trial_v1/outputs.jsonl`, one line per model and cell, with the model and its exact
revision, prompts, raw and cleaned outputs, generation settings, library versions, GPU and date. Download it and
place it at `results/queries/generator_trial_v1/outputs.jsonl` in the repository. The run can be restarted: cells
already done are skipped.

In [ ]:
REPO = "https://github.com/azzs11/heritagerec.git"
BRANCH = "feature/generator-trial"
MODELS = [
    "Qwen/Qwen3-14B",  # Apache-2.0
    "sarvamai/sarvam-m",  # Apache-2.0
]
# Greedy decoding, so a rerun with the same revision and libraries gives the same text. Both model cards
# suggest a low sampling temperature without thinking (Qwen3 0.7, Sarvam-M 0.2); greedy is used for
# reproducibility and recorded here. Thinking mode is off for both: we want the answer, not reasoning.
GENERATION = {"do_sample": False, "max_new_tokens_backstory": 200, "max_new_tokens_query": 64}
OUT_DIR = "generator_trial_v1"

In [ ]:
!git clone --depth 1 -b $BRANCH $REPO heritagerec
!pip install -q -U "transformers>=4.51" accelerate bitsandbytes

In [ ]:
# Optional: a token only raises download rate limits; none of the models is gated.
try:
    from google.colab import userdata
    from huggingface_hub import login

    login(token=userdata.get("HF_TOKEN"))
except Exception as e:
    print("No Hugging Face token used:", type(e).__name__)

In [ ]:
import gc
import json
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import bitsandbytes
import torch
import transformers
from huggingface_hub import HfApi
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

sys.path.insert(0, "heritagerec")
from src.queries.prompts import PROMPT_VERSION, backstory_prompt, query_prompt

cells = [json.loads(line) for line in open("heritagerec/data/queries/generator_trial_cells_v1.jsonl", encoding="utf-8")]
repo_commit = subprocess.run(
    ["git", "-C", "heritagerec", "rev-parse", "HEAD"], capture_output=True, text=True
).stdout.strip()
out_path = Path(OUT_DIR) / "outputs.jsonl"
out_path.parent.mkdir(exist_ok=True)
done = set()
if out_path.exists():
    done = {(r["model"], r["cell_id"]) for r in map(json.loads, out_path.open(encoding="utf-8"))}
environment = {
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "bitsandbytes": bitsandbytes.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "repo_commit": repo_commit,
}
print(len(cells), "cells;", len(done), "outputs already saved;", environment)

In [ ]:
def strip_thinking(text: str) -> str:
    # Thinking is off, but drop an empty or stray think block if a template still emits one.
    return text.split("</think>", 1)[1].strip() if "</think>" in text else text.strip()


def clean_query(text: str) -> str:
    # Keep the first non-empty line and drop wrapping quotes; the raw output is saved too.
    text = strip_thinking(text)
    line = next((x for x in text.splitlines() if x.strip()), "").strip()
    return line.strip("\"'“”‘’`").strip()


def generate(model, tokenizer, prompt: str, max_new_tokens: int) -> str:
    # One user turn only: Gemma's chat template has no system role, so all models get the same shape.
    # return_dict=True gives the token ids and their attention mask in every transformers version
    # (version 5 returns a dict by default; version 4 returned a bare tensor).
    enc = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=GENERATION["do_sample"],
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0, enc["input_ids"].shape[1] :], skip_special_tokens=True).strip()


def run(model_id: str) -> None:
    todo = [c for c in cells if (model_id, c["cell_id"]) not in done]
    if not todo:
        print(model_id, "already done")
        return
    revision = HfApi().model_info(model_id).sha  # load exactly the revision that is recorded
    compute = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=compute)
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, revision=revision, quantization_config=quant, device_map="auto"
    )
    for cell in todo:
        p1 = backstory_prompt(cell)
        backstory_raw = generate(model, tokenizer, p1, GENERATION["max_new_tokens_backstory"])
        backstory = strip_thinking(backstory_raw)
        p2 = query_prompt(cell, backstory)
        raw_query = generate(model, tokenizer, p2, GENERATION["max_new_tokens_query"])
        record = {
            "cell_id": cell["cell_id"],
            "cell": cell,
            "model": model_id,
            "revision": revision,
            "quantization": f"4-bit nf4, compute {str(compute).replace('torch.', '')}",
            "prompt_version": PROMPT_VERSION,
            "backstory_prompt": p1,
            "backstory_raw": backstory_raw,
            "backstory": backstory,
            "query_prompt": p2,
            "query_raw": raw_query,
            "query": clean_query(raw_query),
            "generation": GENERATION,
            "environment": environment,
            "date": datetime.now(timezone.utc).isoformat(),
        }
        with out_path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
        done.add((model_id, cell["cell_id"]))
        print(model_id.split("/")[-1], cell["cell_id"], cell["language"], "|", record["query"])
    del model
    gc.collect()
    torch.cuda.empty_cache()
    # Free the disk for the next model.
    shutil.rmtree(Path.home() / ".cache/huggingface/hub" / f"models--{model_id.replace('/', '--')}", ignore_errors=True)


for model_id in MODELS:
    run(model_id)

Check the printed queries for empty or garbled output. Report any such model as a failure in the trial rather
than rerunning it with different settings.

In [ ]:
print(sum(1 for _ in out_path.open(encoding="utf-8")), "outputs")
try:
    from google.colab import files

    files.download(str(out_path))
except ImportError:
    # On Kaggle the file is in the Output panel (right side) under generator_trial_v1/.
    print("Download", out_path.resolve(), "from the notebook's output files")